# 2026-10-16_3 scTenifoldKnk stage 拆分 target 配对参数汇总表（CSV）

汇总全部 stage × region 拆分轮次（2026-10-15 与 2026-10-16 newTF9）测试过的 TF 对应的 **38 对 TF–target 配对**的关键参数，输出单一 CSV（不含 pooled）：

- 第一轮（11 对）：MEF2C / NFIC / RORB (MG)、ZIC1 (Cone，仅 peripheral)、EGR1 (BC)、ONECUT1 / USF1 (MG)
- 第二轮（27 对，2026-10-16）：KLF9 / ZNF460 / ZNF148 / MAFK / NFE2L1 / KLF15 / THRB / NR2F1 (MG) + E2F6 (Cone，经用户更正)；THRB(var.3) 为上游标注，KO 基因为 THRB
- 阈值口径：多数网络为官方 0.05；EGR1/ONECUT1/USF1 及 ZNF460(yc/oc/yp)/NFE2L1(yp)/KLF15(oc/op) 为放宽 0.01（EGR1 old-central、ONECUT1 young-peripheral 为 0.009）；**不同口径网络间 FC/排名/百分位不可直接比较**
- NA 行：ZIC1 (Cone) central 两侧 KO 未运行（na_reason 注明）；第二轮 14 行 target 不在对应网络检验基因集中（SFRP1-p、ADAMTS7-c、GAS6-p、COTL1-c、ZIC1-c(E2F6)、DIO2-AS1-o-p、ZBTB21-o-c）
- 列定义：`network_threshold` = QC 阈值口径；`n_tested` = 网络内 tested genes 数（排除 TF 自身）；`distance` / `Z` / `FC` / `p.value` / `p.adj` = diffRegulation 原始参数；`FC_rank` / `percentile` = FC 降序排名与百分位（100 = 最强扰动，ties.method="first"）；`significant` = p.adj < 0.05；`direction` / `directionScore` = KO 预测方向（与上游 DEG 方向是两回事，勿混用）
- 输出：`Data/2026-10-16_stage_split_target_pairs_params.csv`（152 行 = 136 数据行 + 16 NA 行）

注意：交接文档 5.3 中 ZIC1→MT2A 的 young/old 数值有笔误，本表按原始 diffRegulation 文件计算（P-Young=3.3 / P-Old=97.0）。

In [ ]:
# ---- 1. 配置：路径 / 配对 / 阈值口径（集中修改处） ----------------------------
suppressPackageStartupMessages({
  library(data.table)
})
set.seed(1)

KNK_DIR    <- "/project/work/2026-10-13_scTenifoldKnk/Data/"
DATE       <- "2026-10-16"
OUT_CSV    <- file.path(KNK_DIR, paste0(DATE, "_stage_split_target_pairs_params.csv"))
PADJ_CUTOFF <- 0.05

# 关注的 TF-target 配对（38 对 = 2026-10-15 轮 11 对 + 2026-10-16 newTF9 轮 27 对；ZIC1 central 两侧 KO 未运行，保留为显式 NA 行）
pair_base <- data.table(
  TF          = c("EGR1", "MEF2C", "MEF2C", "NFIC", "ONECUT1", "ONECUT1", "ONECUT1",
                  "RORB", "USF1", "USF1", "ZIC1",
                  "KLF9", "KLF9", "KLF9", "KLF9",
                  "ZNF460", "ZNF460", "ZNF460",
                  "ZNF148", "ZNF148", "ZNF148",
                  "MAFK", "MAFK", "MAFK", "MAFK",
                  "NFE2L1", "NFE2L1", "NFE2L1", "NFE2L1",
                  "KLF15", "KLF15",
                  "E2F6", "E2F6", "E2F6",
                  "THRB", "THRB", "THRB",
                  "NR2F1"),
  target_gene = c("PDE10A", "APOE", "ABCB11", "DIO2", "DIO2", "TTC39B", "ABCB11",
                  "FNBP1", "TTC39B", "MAN1C1", "MT2A",
                  "ABCB11", "OSBPL10", "CLVS2", "FNBP1",
                  "APOE", "OSBPL10", "PRKCB",
                  "MAN1C1", "SFRP1", "GAS6",
                  "TFRC", "COTL1", "PITPNC1", "ZBTB21",
                  "TFRC", "ADAMTS7", "COTL1", "PITPNC1",
                  "TFRC", "CACNA1D",
                  "ZIC1", "DACH1", "NLGN1",
                  "DIO2-AS1", "MAN1C1", "OSBPL10",
                  "APOE"),
  celltype    = c("BC", rep("MG", 9), "Cone",
                  rep("MG", 20), rep("Cone", 3), rep("MG", 4))
)
group_grid <- CJ(region = c("central", "peripheral"), stage = c("young", "old"))
TARGET_PAIRS <- rbindlist(lapply(seq_len(nrow(pair_base)), function(i) {
  cbind(pair_base[i, .(TF, target_gene, celltype)], group_grid)
}))

# 各 KO 网络的 QC 阈值口径（EGR1/ONECUT1/USF1 及 newTF9 轮部分组合为放宽口径；ZIC1 central 无网络 -> 不标）
NETWORK_MINPCT <- rbindlist(lapply(unique(pair_base$TF), function(tf) {
  g <- group_grid
  if (tf == "ZIC1") g <- g[region == "peripheral"]
  g[, .(TF = tf, region, stage, network_threshold = "0.05")]
}))
NETWORK_MINPCT[TF %in% c("EGR1", "ONECUT1", "USF1"), network_threshold := "0.01"]
NETWORK_MINPCT[TF == "EGR1" & region == "central" & stage == "old", network_threshold := "0.009"]
NETWORK_MINPCT[TF == "ONECUT1" & region == "peripheral" & stage == "young", network_threshold := "0.009"]
# 2026-10-16 newTF9 轮放宽组合（E2F6(Cone) 全组通过 0.05，无需放宽）
NETWORK_MINPCT[TF == "ZNF460" & !(region == "peripheral" & stage == "old"), network_threshold := "0.01"]
NETWORK_MINPCT[TF == "NFE2L1" & region == "peripheral" & stage == "young", network_threshold := "0.01"]
NETWORK_MINPCT[TF == "KLF15" & stage == "old", network_threshold := "0.01"]

# NA 行原因（ZIC1 central：TF 检出率过低，任何阈值均不可行）
NA_REASON <- data.table(
  TF = "ZIC1", celltype = "Cone", region = "central",
  na_reason = "KO not run: TF filtered by scQC (detection 0.19-0.38%, below any feasible minPCT)")

cat("配对 x 组合行数（38 对 x 4 组；含 NA 行）:", nrow(TARGET_PAIRS), "\n")
print(TARGET_PAIRS)

In [ ]:
# ---- 2. 读取各组合 diffRegulation，计算 target 扰动参数并输出 CSV --------------
## 文件发现不限日期前缀；多文件取最新；文件缺失或 target 不在检验基因集 -> NA 行 + na_reason
pair_res <- rbindlist(lapply(seq_len(nrow(TARGET_PAIRS)), function(i) {
  tp <- TARGET_PAIRS[i]
  nr <- NA_REASON[TF == tp$TF & celltype == tp$celltype & region == tp$region, na_reason]
  reason <- if (length(nr)) nr else NA_character_
  f <- list.files(KNK_DIR, pattern = paste0("_", tp$celltype, "_", tp$region, "_",
                                            tp$stage, "_", tp$TF, "_diffRegulation\\.tsv$"),
                  full.names = TRUE)
  if (length(f) > 1) {
    warning(sprintf("%s (%s, %s, %s) 匹配到多个文件，使用最新: %s",
                    tp$TF, tp$celltype, tp$region, tp$stage, basename(sort(f)[length(f)])))
    f <- sort(f)[length(f)]
  }
  d <- NULL
  if (length(f) == 1) {
    d <- fread(f)
    miss <- setdiff(c("gene", "distance", "Z", "FC", "p.value", "p.adj",
                      "direction", "directionScore"), names(d))
    if (length(miss) > 0) stop(basename(f), " 缺少必需列: ", paste(miss, collapse = ", "))
  } else if (is.na(reason)) {
    reason <- "KO not run: no diffRegulation file for this combination"
  }
  ok <- !is.null(d) && tp$target_gene %in% d$gene
  if (!is.null(d) && !ok) {
    reason <- "target gene not in tested genes of this network"
  }
  if (!ok) {
    out <- data.table(network_threshold = NA_character_, n_tested = NA_integer_,
                      distance = NA_real_, Z = NA_real_, FC = NA_real_,
                      p.value = NA_real_, p.adj = NA_real_,
                      FC_rank = NA_integer_, percentile = NA_real_,
                      significant = NA, direction = NA_character_,
                      directionScore = NA_real_, na_reason = reason)
  } else {
    dd <- d[gene != tp$TF]   # 排除被敲除的 TF 自身
    rk <- frank(-dd$FC, ties.method = "first")
    pct <- round(100 * (length(rk) - rk) / pmax(length(rk) - 1, 1), 1)
    j <- which(dd$gene == tp$target_gene)
    thr <- NETWORK_MINPCT[TF == tp$TF & region == tp$region & stage == tp$stage, network_threshold]
    out <- data.table(network_threshold = if (length(thr)) thr else NA_character_,
                      n_tested = length(rk),
                      distance = dd$distance[j], Z = dd$Z[j], FC = dd$FC[j],
                      p.value = dd$p.value[j], p.adj = dd$p.adj[j],
                      FC_rank = as.integer(rk[j]), percentile = pct[j],
                      significant = dd$p.adj[j] < PADJ_CUTOFF,
                      direction = dd$direction[j],
                      directionScore = dd$directionScore[j],
                      na_reason = reason)
  }
  cbind(tp, out)
}))

setcolorder(pair_res, c("TF", "target_gene", "celltype", "region", "stage",
                        "network_threshold", "n_tested", "distance", "Z", "FC",
                        "p.value", "p.adj", "FC_rank", "percentile", "significant",
                        "direction", "directionScore", "na_reason"))
pair_res <- pair_res[order(match(TF, unique(pair_base$TF)), target_gene,
                           match(region, c("central", "peripheral")),
                           match(stage, c("young", "old")))]

fwrite(pair_res, OUT_CSV)
cat("写出:", OUT_CSV, "\n")
print(pair_res)

In [ ]:
# ---- 3. 验证：行数 / 关键数值抽查 / CSV 回读 ----------------------------------
stopifnot(nrow(pair_res) == 152)
stopifnot(sum(!is.na(pair_res$percentile)) == 136)
stopifnot(ncol(pair_res) == 18)

## 与 2026-10-15_2/_4 notebook 已有输出一致的关键数值抽查
g <- function(TF_, tg_, r_, s_) pair_res[TF == TF_ & target_gene == tg_ &
                                            region == r_ & stage == s_]
stopifnot(abs(g("MEF2C", "APOE", "central", "young")$percentile - 98.3) < 1e-9,
          isTRUE(g("MEF2C", "APOE", "central", "young")$significant),
          abs(g("MEF2C", "APOE", "central", "old")$percentile - 100.0) < 1e-9)
stopifnot(abs(g("MEF2C", "ABCB11", "central", "young")$percentile - 98.9) < 1e-9,
          isTRUE(g("MEF2C", "ABCB11", "central", "young")$significant),
          abs(g("MEF2C", "ABCB11", "central", "old")$percentile - 20.9) < 1e-9)
stopifnot(abs(g("ONECUT1", "TTC39B", "central", "young")$percentile - 99.6) < 1e-9,
          isTRUE(g("ONECUT1", "TTC39B", "central", "young")$significant))
stopifnot(abs(g("EGR1", "PDE10A", "peripheral", "young")$percentile - 100.0) < 1e-9,
          isTRUE(g("EGR1", "PDE10A", "peripheral", "young")$significant))

## 2026-10-16 newTF9 轮关键数值抽查（与 2026-10-16_2 输出一致）
stopifnot(abs(g("ZNF460", "APOE", "central", "young")$percentile - 99.3) < 1e-9,
          isTRUE(g("ZNF460", "APOE", "central", "young")$significant),
          abs(g("ZNF460", "APOE", "central", "old")$percentile - 100.0) < 1e-9,
          isTRUE(g("ZNF460", "APOE", "central", "old")$significant))
stopifnot(abs(g("NR2F1", "APOE", "central", "old")$percentile - 100.0) < 1e-9,
          isTRUE(g("NR2F1", "APOE", "central", "old")$significant),
          abs(g("NR2F1", "APOE", "central", "young")$percentile - 98.4) < 1e-9)
stopifnot(abs(g("MAFK", "TFRC", "peripheral", "old")$percentile - 99.8) < 1e-9,
          isTRUE(g("MAFK", "TFRC", "peripheral", "old")$significant),
          abs(g("NFE2L1", "TFRC", "peripheral", "old")$percentile - 99.8) < 1e-9,
          isTRUE(g("NFE2L1", "TFRC", "peripheral", "old")$significant))
stopifnot(abs(g("ZNF460", "PRKCB", "peripheral", "old")$percentile - 99.9) < 1e-9,
          isTRUE(g("ZNF460", "PRKCB", "peripheral", "old")$significant),
          abs(g("ZNF460", "OSBPL10", "central", "young")$percentile - 98.9) < 1e-9,
          isTRUE(g("ZNF460", "OSBPL10", "central", "young")$significant))
## newTF9 轮 NA 行计数（14 行 target 不在检验基因集中）
stopifnot(sum(is.na(pair_res$percentile)) == 16)

## ZIC1->MT2A：按原始文件写入正确方向（P-Young=3.3 / P-Old=97.0，修正交接文档笔误）
stopifnot(abs(g("ZIC1", "MT2A", "peripheral", "young")$percentile - 3.3) < 1e-9,
          abs(g("ZIC1", "MT2A", "peripheral", "old")$percentile - 97.0) < 1e-9)

## ZIC1 central NA 行：数值列全 NA，na_reason 已注明
stopifnot(all(is.na(pair_res[TF == "ZIC1" & region == "central", percentile])),
          all(!is.na(pair_res[TF == "ZIC1" & region == "central", na_reason])))

## CSV 回读核对
csv_back <- fread(OUT_CSV)
stopifnot(nrow(csv_back) == 152, ncol(csv_back) == 18,
          identical(csv_back$TF, pair_res$TF),
          identical(csv_back$target_gene, pair_res$target_gene))

cat("验证通过：152 行（136 数据行 + 16 NA 行）、18 列；关键数值与 2026-10-15_2/_4 及 2026-10-16_2 输出一致；ZIC1->MT2A 为 P-Young=3.3 / P-Old=97.0\n")
print(csv_back[TF == "ZIC1" & target_gene == "MT2A"])